# Maia Lite 0.3.0 — SFT v2.1 Multilingual Balancer

**Objetivo:** preparar até 30.000 exemplos (10.000 PT / 10.000 EN / 10.000 ES), sem inventar traduções, sem misturar validação e treinamento e sem promover traduções não validadas automaticamente.

**Fluxo em duas fases:** (A) preparar seleção e lotes de tradução; (B) importar traduções revisadas e exportar corpus final. Não precisa de GPU. O notebook **não traduz sozinho**: fornece um contrato de arquivos para usar o Maia Dataset Forge ou outro tradutor, com validação humana/LLM independente.

**Importante:** 30.000 linhas não são 30.000 perguntas independentes; versões traduzidas compartilham o mesmo `groupId`. A validação científica existente fica intacta.

## 1 — Montar Drive e definir diretórios

Entradas: `datasets/sft_v2/sft_v2_train.jsonl` e `sft_v2_validation.jsonl`.

Saída: `datasets/sft_v2_1/`. Não altera os arquivos originais.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, csv, hashlib, random, re, unicodedata, collections, math
from datetime import datetime, timezone

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/datasets')
SRC=ROOT/'sft_v2'
OUT=ROOT/'sft_v2_1'
OUT.mkdir(parents=True,exist_ok=True)
TRAIN=SRC/'sft_v2_train.jsonl'; VALID=SRC/'sft_v2_validation.jsonl'
assert TRAIN.is_file() and VALID.is_file(), 'Arquivos sft_v2 ausentes. Confira SRC.'
SEED=20261118
TARGET_PER_LANGUAGE=10000
LANGS=('pt','en','es')
REQUIRE_APPROVAL=True
print('Entradas:', TRAIN, VALID, sep='\n')
print('Saída:',OUT)

## 2 — Utilitários e auditoria das entradas

In [ ]:
def read_jsonl(path):
    with open(path,encoding='utf-8') as f:
        for line_no,line in enumerate(f,1):
            if line.strip():
                try: yield json.loads(line)
                except Exception as e: raise ValueError(f'{path}:{line_no}: {e}') from e

def write_jsonl(path,rows):
    with open(path,'w',encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r,ensure_ascii=False,sort_keys=True)+'\n')

def digest(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

def canonical(s):
    return re.sub(r'\s+',' ',unicodedata.normalize('NFKC',str(s))).strip().casefold()

def lang_counts(rows): return dict(collections.Counter(x['language'] for x in rows))

def ensure_records(rows):
    required={'id','groupId','language','prompt','completion','category','source'}
    for i,r in enumerate(rows):
        missing=required-set(r)
        assert not missing, f'Registro {i} sem {missing}'
        assert r['language'] in LANGS
        assert str(r['prompt']).strip() and str(r['completion']).strip()

train=list(read_jsonl(TRAIN)); val=list(read_jsonl(VALID))
ensure_records(train);ensure_records(val)
assert len({r['id'] for r in train+val})==len(train)+len(val)
train_groups={r['groupId'] for r in train}
val_groups={r['groupId'] for r in val}
assert train_groups.isdisjoint(val_groups),'groupId compartilhado entre treino e validação'
assert {canonical(r['prompt']) for r in train}.isdisjoint({canonical(r['prompt']) for r in val}), 'Prompt compartilhado entre treino e validação'
print('Treino:',len(train),lang_counts(train))
print('Validação:',len(val),lang_counts(val))
print('Grupos treino:',len(train_groups),'grupos validação:',len(val_groups))

## 3 — Seleção dos exemplos originais

Preserva todos os exemplos PT/ES enquanto houver capacidade. Para inglês, seleciona até 10.000 com amostragem reprodutível e diversidade por categoria. Mantém a ligação `groupId` e não move registros de validação. Excedentes ficam registrados separadamente.

A seleção de inglês usa prioridade para grupos que ainda não têm versão PT/ES, aumentando a cobertura potencial de tradução. O conteúdo científico trilíngue curado permanece no treinamento.

In [ ]:
bylang={l:[dict(r) for r in train if r['language']==l] for l in LANGS}
assert all(len(bylang[l])>=1 for l in LANGS)
assert all(len(bylang[l])<=TARGET_PER_LANGUAGE for l in ('pt','es')), 'PT/ES excedem meta; revise seleção manualmente'

selected={l:list(bylang[l]) for l in ('pt','es')}
other_lang_groups={r['groupId'] for l in ('pt','es') for r in selected[l]}
eng=bylang['en']
# Diversifica por categoria via round-robin e prioriza grupos sem tradução existente.
buckets=collections.defaultdict(list)
for r in eng:
    priority=0 if r['groupId'] not in other_lang_groups else 1
    buckets[(priority,r['category'])].append(r)
for k,rows in buckets.items():
    rows.sort(key=lambda r:r['id'])
    random.Random(SEED+int(hashlib.sha256(str(k).encode()).hexdigest()[:8],16)).shuffle(rows)
eng_selected=[]
for priority in (0,1):
    cats=sorted(k for k in buckets if k[0]==priority)
    while len(eng_selected)<TARGET_PER_LANGUAGE and any(buckets[k] for k in cats):
        for k in cats:
            if buckets[k] and len(eng_selected)<TARGET_PER_LANGUAGE:
                eng_selected.append(buckets[k].pop())
selected['en']=eng_selected
assert len(selected['en'])==TARGET_PER_LANGUAGE, 'Não há 10.000 exemplos EN disponíveis'
selected_all=[r for l in LANGS for r in selected[l]]
selected_ids={r['id'] for r in selected_all}
reserve=[r for r in train if r['id'] not in selected_ids]
write_jsonl(OUT/'selected_originals.jsonl',selected_all)
write_jsonl(OUT/'reserve_originals.jsonl',reserve)
print('Originais selecionados:',lang_counts(selected_all))
print('Reserva:',len(reserve))

## 4 — Gerar tarefas de tradução (sem criar respostas fictícias)

Para cada lacuna em PT e ES, o notebook procura um exemplo EN do treinamento cujo grupo ainda não possua versão naquele idioma. Gera tarefas independentes para **prompt e completion**.

Os campos `prompt_source` e `completion_source` são o texto integral; o tradutor deve preservar instruções, fórmulas, números, alternativas e `####` do GSM8K. A saída deverá conter `translated_prompt`, `translated_completion`, `approved=true` e `reviewer`. Não use a validação como fonte de tradução.

In [ ]:
selected_by_group=collections.defaultdict(dict)
for r in selected_all:
    g=selected_by_group[r['groupId']]
    g.setdefault(r['language'],r)

jobs=[]; deficits={l:TARGET_PER_LANGUAGE-len(selected[l]) for l in ('pt','es')}
for lang in ('pt','es'):
    # Para ciência trilíngue, não retraduzir um grupo já presente no idioma.
    # Para OASST, groupId é a árvore de conversa, podendo conter várias perguntas distintas.
    candidates=[r for r in selected['en'] if not (r['category']=='science_custom' and lang in selected_by_group[r['groupId']])]
    # Maior diversidade por categoria, ordem reprodutível.
    cat=collections.defaultdict(list)
    for r in candidates:cat[r['category']].append(r)
    for k in cat:random.Random(SEED+len(k)+(1 if lang=='pt' else 2)).shuffle(cat[k])
    ordered=[]; keys=sorted(cat)
    while any(cat.values()):
        for k in keys:
            if cat[k]:ordered.append(cat[k].pop())
    for r in ordered[:deficits[lang]]:
        jobs.append({
            'job_id':f'translate::{r["id"]}::{lang}',
            'source_id':r['id'],'groupId':r['groupId'],
            'source_language':'en','target_language':lang,
            'category':r['category'],'source':r['source'],
            'license':r.get('license','unknown'),
            'prompt_source':r['prompt'],
            'completion_source':r['completion'],
            'translated_prompt':'','translated_completion':'',
            'approved':False,'reviewer':'',
            'review_notes':''
        })

job_path=OUT/'translation_jobs.jsonl'
write_jsonl(job_path,jobs)
with open(OUT/'translation_jobs.csv','w',newline='',encoding='utf-8-sig') as f:
    w=csv.DictWriter(f,fieldnames=list(jobs[0]) if jobs else ['job_id'])
    w.writeheader();w.writerows(jobs)
print('Déficits:',deficits)
print('Tarefas exportadas:',len(jobs),dict(collections.Counter(j['target_language'] for j in jobs)))
print('Arquivo:',job_path)
for l in ('pt','es'):
    available=sum(j['target_language']==l for j in jobs)
    if available<deficits[l]: print('AVISO:',l,'faltam',deficits[l]-available,'fontes elegíveis; serão necessários novos exemplos nativos.')

## 5 — Tradução e revisão externa

**Etapa manual/externa obrigatória.** Use `translation_jobs.jsonl` no Maia Dataset Forge (ou outro sistema de tradução). Traduza **prompt e resposta**, revise com um segundo modelo e aprove somente os itens semanticamente equivalentes. Salve o resultado como:

`datasets/sft_v2_1/translation_reviewed.jsonl`

Cada linha deve manter `job_id` e conter `translated_prompt`, `translated_completion`, `approved` (booleano verdadeiro), `reviewer` (não vazio). Pode devolver apenas os itens aprovados; itens não aprovados não entram no corpus. Para manter a auditabilidade, não sobrescreva `translation_jobs.jsonl`.

Exemplo de um item aprovado (campos ilustrativos):

```json
{"job_id":"translate::exemplo::pt","translated_prompt":"Pergunta traduzida...","translated_completion":"Resposta traduzida...","approved":true,"reviewer":"validation-14b-v1"}
```

**Não execute as células finais esperando 30 mil exemplos enquanto não tiver o arquivo revisado.** Elas produzirão uma versão parcial claramente identificada.

## 6 — Importar traduções aprovadas e validar integridade

As validações automáticas verificam origem, idioma, duplicatas, campos obrigatórios, identidade de grupo e sobreposição exata. Elas **não comprovam fidelidade semântica**; isso depende da revisão externa.

In [ ]:
reviewed_path=OUT/'translation_reviewed.jsonl'
reviewed=list(read_jsonl(reviewed_path)) if reviewed_path.exists() else []
job_map={j['job_id']:j for j in jobs}
seen_jobs=set(); accepted=[]; rejected=collections.Counter()
heldout_prompts={canonical(r['prompt']) for r in val}
used_prompts={canonical(r['prompt']) for r in selected_all}
used_ids={r['id'] for r in selected_all+val}
for r in reviewed:
    jid=r.get('job_id')
    if jid not in job_map:rejected['unknown_job']+=1;continue
    if jid in seen_jobs:rejected['duplicate_job']+=1;continue
    seen_jobs.add(jid)
    j=job_map[jid]
    if r.get('approved') is not True or not str(r.get('reviewer','')).strip():
        rejected['not_approved_or_no_reviewer']+=1;continue
    p=str(r.get('translated_prompt','')).strip()
    a=str(r.get('translated_completion','')).strip()
    if not p or not a:rejected['empty_translation']+=1;continue
    if len(p)>3500 or len(a)>3500 or len(p)<12:rejected['length']+=1;continue
    if canonical(p) in heldout_prompts:rejected['heldout_prompt']+=1;continue
    if canonical(p) in used_prompts:rejected['duplicate_prompt']+=1;continue
    # Metadados são herdados do registro original, não do arquivo de tradução.
    src=next(x for x in selected['en'] if x['id']==j['source_id'])
    out=dict(src)
    out['id']=f'{src["id"]}::translated::{j["target_language"]}'
    if out['id'] in used_ids:rejected['duplicate_id']+=1;continue
    out['language']=j['target_language']
    out['prompt']=p;out['completion']=a
    out['translation_of']=src['id']
    out['translation_job_id']=jid
    out['translation_reviewer']=str(r['reviewer'])
    out['translation_review_notes']=str(r.get('review_notes',''))
    out['translation_status']='approved'
    out['translation_method']=str(r.get('translation_method','external'))
    used_prompts.add(canonical(p));used_ids.add(out['id'])
    accepted.append(out)
print('Linhas revisadas:',len(reviewed),'aprovadas e aceitas:',len(accepted),'rejeitadas:',dict(rejected))
print('Traduções por idioma:',lang_counts(accepted))

## 7 — Exportar corpus, validação intacta, auditoria e manifest

In [ ]:
final_train=selected_all+accepted
random.Random(SEED+100).shuffle(final_train)
final_val=list(val)
assert {r['groupId'] for r in final_train}.isdisjoint({r['groupId'] for r in final_val})
assert {canonical(r['prompt']) for r in final_train}.isdisjoint({canonical(r['prompt']) for r in final_val})
assert len({r['id'] for r in final_train+final_val})==len(final_train)+len(final_val)
assert len(final_val)==len(val)
counts=collections.Counter(r['language'] for r in final_train)
assert all(counts[l]<=TARGET_PER_LANGUAGE for l in LANGS)
assert all(r['language'] in LANGS for r in final_train)

train_out=OUT/'sft_v2_1_train.jsonl'
val_out=OUT/'sft_v2_1_validation.jsonl'
write_jsonl(train_out,final_train);write_jsonl(val_out,final_val)
manifest={
 'project':'Maia Lite 0.3.0','pipeline':'SFT v2.1 multilingual balancer',
 'created_utc':datetime.now(timezone.utc).isoformat(),'seed':SEED,
 'status':'READY_FOR_REVIEW' if all(counts[l]==TARGET_PER_LANGUAGE for l in LANGS) else 'INCOMPLETE_TRANSLATIONS',
 'target_train':TARGET_PER_LANGUAGE*3,'target_per_language':TARGET_PER_LANGUAGE,
 'actual_train':len(final_train),'actual_validation':len(final_val),
 'train_by_language':dict(counts),'train_by_category':dict(collections.Counter(r['category'] for r in final_train)),
 'validation_by_language':lang_counts(final_val),
 'shortfall_by_language':{l:TARGET_PER_LANGUAGE-counts[l] for l in LANGS},
 'translation_jobs':len(jobs),'reviewed_lines':len(reviewed),'accepted_translations':len(accepted),
 'rejected_translations':dict(rejected),'reserved_originals':len(reserve),
 'distinct_train_groups':len({r['groupId'] for r in final_train}),
 'input_sha256':{p.name:digest(p) for p in (TRAIN,VALID)},
 'output_sha256':{p.name:digest(p) for p in (train_out,val_out,job_path)},
 'warnings':[
   'No translation is automatically accepted without approved=true and reviewer.',
   'Automated checks do not verify semantic fidelity, factuality, licenses or safety.',
   'The 273 curated scientific validation examples are not promoted to training.',
   'Source licenses, including ARC share-alike and per-paper rights, require review.',
   'Multilingual translations are not independent semantic questions.',
   'Benchmark overlap and semantic near-duplicates require separate audit.'
 ]
}
(OUT/'manifest_v2_1.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps({k:manifest[k] for k in ['status','actual_train','actual_validation','train_by_language','shortfall_by_language','accepted_translations']},ensure_ascii=False,indent=2))
print('Salvo em:',OUT)

## 8 — Critérios antes do LoRA v2

- Conferir `manifest_v2_1.json` e se o status é `READY_FOR_REVIEW`.
- Auditar uma amostra estratificada por idioma/categoria; checar licenças e equivalência matemática.
- Manter `sft_v2_1_validation.jsonl` isolado, sem usar suas questões na tradução.
- Para comparação justa com o LoRA científico v1, iniciar novamente do checkpoint base de **300.000 steps**.
- **Não** interpretar 30.000 pares trilíngues como 30.000 questões independentes.

Caso não haja traduções revisadas, o notebook termina com `INCOMPLETE_TRANSLATIONS`: isso é esperado e evita que exemplos não validados sejam usados no treinamento.